Libraries and Setup

In [7]:
!pip install scikit-learn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

if not os.path.exists('models'):
    os.makedirs('models')
    
print("Libraries imported and model directory created.")

Libraries imported and model directory created.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Data Loading & Advanced Feature

In [8]:
FILE_PATH = 'drugs.csv'

if os.path.exists(FILE_PATH):
    df = pd.read_csv(FILE_PATH)
    print("Data loaded successfully")

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])

        df = df.sort_values(by=['Brand_Name', 'Date'])

        df['Month'] = df['Date'].dt.month
        df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
        df['Year'] = df['Date'].dt.year


        for l in [1, 2, 3, 6, 12]:
            df[f'Lag_{l}'] = df.groupby('Brand_Name')['Issued_Qty'].shift(l)

        df['Rolling_Mean_3'] = df.groupby('Brand_Name')['Issued_Qty'].transform(lambda x: x.rolling(window=3).mean())
        df['Rolling_Mean_6'] = df.groupby('Brand_Name')['Issued_Qty'].transform(lambda x: x.rolling(window=6).mean())
        
        df = df.dropna()
        print("Hybrid Lags, Rolling Features, and Cyclical Months created")
    
    display(df.head())
else:
    print(f"Error: {FILE_PATH} not found")

Data loaded successfully
Hybrid Lags, Rolling Features, and Cyclical Months created


,Date,Drug_Name,Brand_Name,Strength,Opening_Stock,Received_Qty,Issued_Qty,Closing_Stock,Shortage_Flag,Lead_Time,...,Month_Sin,Month_Cos,Year,Lag_1,Lag_2,Lag_3,Lag_6,Lag_12,Rolling_Mean_3,Rolling_Mean_6
742,2021-01-01,Pioglitazone,Actos,15mg,205.0,148.0,141.0,212.0,0,9.0,...,0.500000,8.660254e-01,2021,144.0,181.0,145.0,179.0,132.0,155.333333,178.833333
743,2021-02-01,Pioglitazone,Actos,15mg,212.0,186.0,177.0,221.0,0,10.0,...,0.866025,5.000000e-01,2021,141.0,144.0,181.0,273.0,187.0,154.000000,162.833333
744,2021-03-01,Pioglitazone,Actos,15mg,221.0,169.0,161.0,229.0,0,12.0,...,1.000000,6.123234e-17,2021,177.0,141.0,144.0,189.0,147.0,159.666667,158.166667
745,2021-04-01,Pioglitazone,Actos,15mg,229.0,184.0,175.0,238.0,0,8.0,...,0.866025,-5.000000e-01,2021,161.0,177.0,141.0,145.0,199.0,171.000000,163.166667
746,2021-05-01,Pioglitazone,Actos,15mg,238.0,185.0,176.0,247.0,0,11.0,...,0.500000,-8.660254e-01,2021,175.0,161.0,177.0,181.0,158.0,170.666667,162.333333


Missing Value Handling

In [9]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

text_cols = df.select_dtypes(include=[object]).columns
df[text_cols] = df[text_cols].fillna('Unknown')

print("Missing values handled successfully")

Missing values handled successfully


Label Encoding for Global Panel Model

In [10]:
le = LabelEncoder()

if 'Brand_Name' in df.columns:

    df['Brand_Name_Encoded'] = le.fit_transform(df['Brand_Name'])
    joblib.dump(le, 'models/label_encoder.pkl')
    
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print("Drug Brand Mapping created for Global Model")
else:
    print("Error: Brand_Name column not found")

Drug Brand Mapping created for Global Model


Log Transformation, Scaling & Time-Series Split

In [11]:
from sklearn.model_selection import train_test_split


drop_cols = [
    'Date', 'Drug_Name', 'Brand_Name', 'Strength', 'Month',
    'Current_Stock', 'Received_Qty', 'Lead_Time', 'Shortage_Flag',
    'Opening_Stock', 'Closing_Stock', 'Issued_Qty'
]

df['Issued_Qty_Log'] = np.log1p(df['Issued_Qty'])
target_col = 'Issued_Qty_Log'

X = df.drop(columns=[target_col] + drop_cols, errors='ignore')
y = df[target_col]

X = X.select_dtypes(include=[np.number])

print("Features used for training (Global Panel Model):")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

joblib.dump(scaler_X, 'models/scaler_X.pkl')
joblib.dump(scaler_y, 'models/scaler_y.pkl')

np.save('models/X_train_scaled.npy', X_train_scaled)
np.save('models/X_test_scaled.npy', X_test_scaled)
np.save('models/y_train_scaled.npy', y_train_scaled)
np.save('models/y_test_scaled.npy', y_test_scaled)

X_scaled_full = np.vstack((X_train_scaled, X_test_scaled))
y_scaled_full = np.vstack((y_train_scaled, y_test_scaled))
np.save('models/X_scaled.npy', X_scaled_full)
np.save('models/y_scaled.npy', y_scaled_full)

print("\n[SUCCESS] Preprocessing complete with 5-Point Optimization")
print(f"Total Features: {X.shape[1]} | Total Records: {len(df)}")

Features used for training (Global Panel Model):
['USD_Rate', 'Month_Sin', 'Month_Cos', 'Year', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_6', 'Lag_12', 'Rolling_Mean_3', 'Rolling_Mean_6', 'Brand_Name_Encoded']

[SUCCESS] Preprocessing complete with 5-Point Optimization
Total Features: 12 | Total Records: 1830
